# Train the "Hey Huncho" wake word

Trains a custom openWakeWord-compatible ONNX model using [livekit-wakeword](https://github.com/livekit/livekit-wakeword) — fully synthetic data (local Piper TTS), no API keys.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.

At the end, `huncho.onnx` downloads automatically → drop it into `huncho/assets/wake/` and restart Huncho.

In [ ]:
# 1) System deps (Colab is Ubuntu)
!apt-get -qq update && apt-get -qq install -y espeak-ng libsndfile1 ffmpeg sox > /dev/null
print('system deps OK')

In [ ]:
# 2) livekit-wakeword with training extras (needs Python 3.11+ — Colab default is fine)
import sys; print(sys.version)
%pip install -q "livekit-wakeword[train,eval,export]"
!livekit-wakeword --help | head -30

In [ ]:
# 3) Write the training config
import pathlib
config = '''\
model_name: huncho
target_phrases:
  - "hey huncho"
  - "huncho"
n_samples: 10000
model:
  model_type: conv_attention
  model_size: small
steps: 50000
target_fp_per_hour: 0.2
output_dir: ./out
'''
pathlib.Path('huncho.yaml').write_text(config)
print(config)
# If the run cell below complains about unknown config keys, check the schema:
# !livekit-wakeword run --help

In [ ]:
# 4) Download TTS/augmentation assets, then run the full pipeline:
#    synthesize samples -> augment -> train -> export ONNX
#    (T4 GPU: expect roughly 30–90 min total)
!livekit-wakeword setup --config huncho.yaml
!livekit-wakeword run huncho.yaml

In [ ]:
# 5) Find the exported ONNX and download it as huncho.onnx
import glob, shutil
candidates = glob.glob('**/*.onnx', recursive=True)
print('found:', candidates)
src = [c for c in candidates if 'huncho' in c.lower()] or candidates
shutil.copy(src[0], 'huncho.onnx')
from google.colab import files
files.download('huncho.onnx')